# SLM — Built from Scratch
A 162M parameter GPT-style language model built entirely from scratch
in PyTorch, pretrained on WikiText-2.

**Architecture:** 12 transformer blocks · 12 attention heads · 768 embedding dims · 162M parameters  
**Results:** Validation Loss 2.11 · Perplexity 8.25

In [1]:
!pip install transformers datasets torch -q

## 1. Imports
Core libraries: PyTorch for model building and training,
HuggingFace Transformers for the pretrained tokenizer.

In [2]:
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer

## 2. Tokenizer
GPT-2's pretrained Byte Pair Encoding (BPE) tokenizer with a vocabulary
of 50,257 tokens. BPE splits rare words into subword units (e.g. "Messi"
→ "Mess" + "i") while common words get their own token. Reusing GPT-2's
tokenizer keeps focus on the model architecture.

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("Vocabulary size:", tokenizer.vocab_size)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Vocabulary size: 50257


In [4]:
text = "Messi scored an amazing goal"

# Encode — text to numbers
tokens = tokenizer.encode(text)
print("Tokens:", tokens)

# Decode — numbers back to text
decoded = tokenizer.decode(tokens)
print("Decoded:", decoded)

# See what each token actually is
for token in tokens:
    print(f"{token} → '{tokenizer.decode([token])}'")

Tokens: [36479, 72, 7781, 281, 4998, 3061]
Decoded: Messi scored an amazing goal
36479 → 'Mess'
72 → 'i'
7781 → ' scored'
281 → ' an'
4998 → ' amazing'
3061 → ' goal'


## 3. Pretraining Dataset — WikiText-2
WikiText-2 is a cleaned subset of Wikipedia (~2MB, 36K examples).
Small enough to train on free Colab GPUs while still teaching the
model general English grammar and structure.

In [5]:
from datasets import load_dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

print(dataset)
print("\nSample text:")
print(dataset["train"][5]["text"])

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Sample text:
 It met with positive sales in Japan , and was praised by both Japanese and western critics . After release , it received downloadable content , along with an expanded edition in November of that year . It was also adapted into manga and an original video animation series . Due to low sales of Valkyria Chronicles II , Valkyria Chronicles III was not localized , but a fan translation compatible with the game 's expanded edition was released in 2014 . Media.Vision would return to the franchise with the development of Valkyria : Azure Revolution for the PlayStation 4 . 



## 4. Tokenization Pipeline
Each text sequence is tokenized, truncated or padded to 128 tokens,
and converted to integer IDs. Dataset mapped in batches for efficiency,
producing input_ids ready for the model.

In [6]:
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
        return_tensors="pt"
    )

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized)

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 3760
    })
})


## 5. Dataset and DataLoader
Implements the causal language modeling objective — each training
example is an (x, y) pair where y is x shifted one position right.
Teaches next token prediction: given all previous tokens, predict
the next one. One sequence of 127 tokens yields 127 independent
training examples. Batched into groups of 16 via DataLoader.

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, tokenized_data):
        self.input_ids = tokenized_data["input_ids"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.input_ids[idx])

        # input = all tokens except last
        # target = all tokens except first (shifted by 1)
        # this is how language models train — predict next token
        x = input_ids[:-1]
        y = input_ids[1:]

        return x, y

train_dataset = TextDataset(tokenized["train"])
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


x, y = train_dataset[0]

for i in range(len(train_dataset)):
    x, y = train_dataset[i]
    if x[0].item() != 50256:
        print(f"Found real content at index {i}")
        print("\nInput tokens:", x[:5])
        print("Target tokens:", y[:5])

        # Decode to see actual words
        print("\nInput text:", tokenizer.decode(x[:5]))
        print("Target text:", tokenizer.decode(y[:5]))
        break

Found real content at index 1

Input tokens: tensor([  796,   569, 18354,  7496, 17740])
Target tokens: tensor([  569, 18354,  7496, 17740,  6711])

Input text:  = Valkyria Chronicles
Target text:  Valkyria Chronicles III


## 6. Embeddings
Two learned lookup tables — token embeddings (50257 × 768) encode
what each token means, positional embeddings (127 × 768) encode
where each token sits in the sequence. Added elementwise so each
token's final vector carries both meaning and position.  
Output shape: (batch=16, seq_len=127, embed_dim=768).

In [8]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_seq_len, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(max_seq_len, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device)
        positions = positions.unsqueeze(0).expand(batch_size, -1)
        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)
        out = self.dropout(token_emb + pos_emb)
        return out

# test it
vocab_size = tokenizer.vocab_size  # 50257
embed_dim = 768
max_seq_len = 127

emb = Embeddings(vocab_size, embed_dim, max_seq_len)

x, y = next(iter(train_loader))
print("Input batch shape:", x.shape)
out = emb(x)
print("Embedding output shape:", out.shape)

Input batch shape: torch.Size([16, 127])
Embedding output shape: torch.Size([16, 127, 768])


## 7. Single Attention Head
Core of the transformer. Input is projected into Query, Key and Value
vectors via three learned linear transformations (768 → 64 dims each).
Scores computed as QKᵀ/√64. A causal mask blocks future tokens,
preventing the model from attending to positions it shouldn't know
about during training. Softmax converts scores to weights, producing
a weighted sum of Values. Output: (16, 127, 64).

In [9]:
class AttentionHead(nn.Module):
  def __init__(self,embed_dim,head_dim):
    super().__init__()
    self.W_q = nn.Linear(embed_dim,head_dim,bias = False)
    self.W_k = nn.Linear(embed_dim,head_dim,bias = False)
    self.W_v = nn.Linear(embed_dim,head_dim,bias = False)
    self.scale = head_dim ** -0.5
  def forward(self,x):
    # x shape: (batch_size(16), seq_len(127), embed_dim(768))
    Q = self.W_q(x)
    K = self.W_k(x)
    V = self.W_v(x)
    '''
    W_q shape:  (768, 64)   [internal matrix of the linear layer]
    output shape: (16, 127, 64)
    '''

    scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
    '''
    `torch.matmul(Q, K_transposed)`:

    Q shape:      (16, 127, 64)
    K^T shape:    (16, 64,  127)
    result shape: (16, 127, 127)
    This gives every token a score against every other token. The (127×127) matrix means:
    row i, col j = how much token i should attend to token j
    '''
    seq_len = x.size(1)
    mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
    scores = scores.masked_fill(mask, float('-inf'))
    '''
Wherever mask is `True`, replace that score with `-inf`:
          tok0   tok1   tok2   tok3   tok4
tok0   [  0.8   -inf   -inf   -inf   -inf ]  ← tok0 only sees itself
tok1   [  0.5    0.7   -inf   -inf   -inf ]  ← tok1 sees tok0, tok1
tok2   [  0.3    0.4    0.9   -inf   -inf ]  ← tok2 sees tok0,1,2
tok3   [  0.1    0.2    0.3    0.8   -inf ]
tok4   [  0.3    0.5    0.2    0.4    0.6 ]  ← tok4 sees everything
     '''
    # softmax to get attention weights
    weights = torch.softmax(scores, dim=-1)

    '''
    weights shape: (batch_size(16), seq_len(127), seq_len(127))
    Converts raw scores into probabilities along the last dimension
tok0: [1.0,  0.0,  0.0,  0.0,  0.0]  ← 100% attention to itself
tok1: [0.4,  0.6,  0.0,  0.0,  0.0]  ← split between tok0 and tok1
tok2: [0.2,  0.3,  0.5,  0.0,  0.0]  ← split between tok0, 1, 2
    '''

    out = torch.matmul(weights, V)
    # weighted sum of values ,out shape: (batch_size, seq_len, head_dim)

    return out

In [10]:
# test one attention head
embed_dim = 768
head_dim = 64   # 768 / 12 heads

head = AttentionHead(embed_dim, head_dim)

# use embedding output
out = emb(x)                    # (16, 127, 768)
head_out = head(out)
print("Attention head output:", head_out.shape)

Attention head output: torch.Size([16, 127, 64])


## 8. Multi-Head Attention
Runs 12 attention heads in parallel via nn.ModuleList, each
specializing in different token relationships. Outputs concatenated
back to 768 dims and passed through final projection W_O to mix
information across all heads.  
Output shape: (16, 127, 768).

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads  # 768 // 12 = 64

        # create 12 attention heads
        self.heads = nn.ModuleList([
            AttentionHead(embed_dim, self.head_dim)
            for _ in range(num_heads)
        ])

        # final projection to mix all heads together
        self.W_o = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # run all 12 heads in parallel
        head_outputs = [head(x) for head in self.heads]
        # each head output: (16, 127, 64)

        # concatenate all heads along last dimension
        x = torch.cat(head_outputs, dim=-1)
        # shape: (16, 127, 768)  ← 12 × 64 = 768, back to original

        '''
        final linear projection
        After concatenating 12 heads you have 768 dims again, but they're just 12 separate chunks sitting side by side.
        W_o mixes information across all heads — lets them communicate and produce one unified representation.
        '''
        x = self.W_o(x)
        # shape: (16, 127, 768)

        return x

In [12]:
embed_dim = 768
num_heads = 12

mha = MultiHeadAttention(embed_dim, num_heads)

out = emb(x)                    # (16, 127, 768)
mha_out = mha(out)
print("MHA output shape:", mha_out.shape)

MHA output shape: torch.Size([16, 127, 768])


## 9. Feed Forward Network
Two-layer MLP applied independently to each token position.
Expands 768 → 3072 dims (4× expansion), applies GELU non-linearity,
compresses back to 768. Attention handles relationships between tokens —
FFN handles per-token computation and knowledge storage.

In [13]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),  # 768 → 3072
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),  # 3072 → 768
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [14]:
ff = FeedForward(embed_dim=768)

out = emb(x)               # (16, 127, 768)
ff_out = ff(out)
print("FFN output shape:", ff_out.shape)

FFN output shape: torch.Size([16, 127, 768])


## 10. Transformer Block
Assembles Multi-Head Attention and FFN with pre-norm architecture —
LayerNorm applied before each sublayer for training stability.
Residual connections (x = x + sublayer(x)) provide direct gradient
highways during backpropagation, letting each block learn small
refinements rather than full transformations.  
Same shape in and out: (16, 127, 768).

In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.ff = FeedForward(embed_dim, dropout)

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # attention sub-layer
        residual = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = residual + x          # residual connection 1

        # ffn sub-layer
        residual = x
        x = self.norm2(x)
        x = self.ff(x)
        x = residual + x          # residual connection 2

        return x

In [16]:
block = TransformerBlock(embed_dim=768, num_heads=12)

out = emb(x)                      # (16, 127, 768)
block_out = block(out)
print("Block output shape:", block_out.shape)  # (16, 127, 768)

Block output shape: torch.Size([16, 127, 768])


## 11. Full Model
Stacks 12 transformer blocks sequentially, followed by a final
LayerNorm and a language model head (768 → 50257) projecting to
vocabulary logits. Total: 162M parameters. The extra params vs
the 125M target come from the lm_head alone (768 × 50257 ≈ 38M).

In [17]:
class GPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, max_seq_len, dropout=0.1):
        super().__init__()

        self.embedding = Embeddings(vocab_size, embed_dim, max_seq_len, dropout)

        # stack 12 transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        # final projection → vocabulary
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, x):
        # x shape: (16, 127)

        x = self.embedding(x)
        # (16, 127, 768)

        for block in self.blocks:
            x = block(x)
        # (16, 127, 768)

        x = self.norm(x)
        # (16, 127, 768)

        logits = self.lm_head(x)
        # (16, 127, 50257)

        return logits

In [18]:
model = GPT(
    vocab_size=50257,
    embed_dim=768,
    num_heads=12,
    num_layers=12,
    max_seq_len=127,
    dropout=0.1
)

# count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# test forward pass
x, y = next(iter(train_loader))
logits = model(x)
print(f"Logits shape: {logits.shape}")  # (16, 127, 50257)

Total parameters: 162,320,640
Logits shape: torch.Size([16, 127, 50257])


## 12. Training Loop
Trained for 3 epochs on WikiText-2 using AdamW optimizer
(lr=3e-4, weight_decay=0.01) and CrossEntropyLoss. Gradient
clipping at norm 1.0 prevents exploding gradients.  
Loss dropped from 11.45 (random initialization — equivalent to
log(50257)) to avg 1.78 by epoch 3.

In [19]:
import torch.optim as optim

# hyperparameters
EPOCHS = 3
LEARNING_RATE = 3e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Training on: {DEVICE}")

# move model to GPU if available
model = model.to(DEVICE)

# optimizer — Adam is standard for transformers
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# loss function
criterion = nn.CrossEntropyLoss()

# training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, (x, y) in enumerate(train_loader):
        # move batch to GPU
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        # forward pass
        logits = model(x)
        # logits: (16, 127, 50257)

        # reshape for loss calculation
        logits = logits.view(-1, 50257)   # (16×127, 50257) = (2032, 50257)
        y = y.view(-1)                     # (16×127,)       = (2032,)

        # compute loss
        loss = criterion(logits, y)

        # backward pass
        optimizer.zero_grad()
        loss.backward()

        # gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # update weights
        optimizer.step()

        total_loss += loss.item()

        # print progress every 100 batches
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} complete | Avg Loss: {avg_loss:.4f}\n")

print("Training complete!")

Training on: cuda
Epoch 1 | Batch 0 | Loss: 11.4541
Epoch 1 | Batch 100 | Loss: 2.9335
Epoch 1 | Batch 200 | Loss: 2.9678
Epoch 1 | Batch 300 | Loss: 3.7483
Epoch 1 | Batch 400 | Loss: 2.8398
Epoch 1 | Batch 500 | Loss: 2.1961
Epoch 1 | Batch 600 | Loss: 2.5522
Epoch 1 | Batch 700 | Loss: 1.0372
Epoch 1 | Batch 800 | Loss: 3.2046
Epoch 1 | Batch 900 | Loss: 1.8681
Epoch 1 | Batch 1000 | Loss: 1.8548
Epoch 1 | Batch 1100 | Loss: 2.8594
Epoch 1 | Batch 1200 | Loss: 1.8699
Epoch 1 | Batch 1300 | Loss: 1.7597
Epoch 1 | Batch 1400 | Loss: 1.7572
Epoch 1 | Batch 1500 | Loss: 1.7688
Epoch 1 | Batch 1600 | Loss: 1.7328
Epoch 1 | Batch 1700 | Loss: 1.4756
Epoch 1 | Batch 1800 | Loss: 0.7811
Epoch 1 | Batch 1900 | Loss: 1.1477
Epoch 1 | Batch 2000 | Loss: 3.0998
Epoch 1 | Batch 2100 | Loss: 1.4621
Epoch 1 | Batch 2200 | Loss: 1.8772

Epoch 1 complete | Avg Loss: 2.3305

Epoch 2 | Batch 0 | Loss: 2.2626
Epoch 2 | Batch 100 | Loss: 2.8531
Epoch 2 | Batch 200 | Loss: 2.7339
Epoch 2 | Batch 300 | Lo

## 13. Text Generation — Greedy Decoding
Generates text by repeatedly predicting the next token using argmax.
Simple but prone to repetitive loops. EOS token check prevents
infinite generation.

In [20]:
def generate(model, tokenizer, prompt, max_new_tokens=50, device=DEVICE):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    eos_token_id = tokenizer.eos_token_id

    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_cropped = input_ids[:, -127:]
            logits = model(input_cropped)
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            # stop if end of text token
            if next_token.item() == eos_token_id:
                break

            input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0])

output = generate(model, tokenizer, "The football match ended")
print(output)

The football match ended in the first time since the second quarter of the quarter of the season , having scored the first time in the second quarter in the second quarter of the first time since the second quarter of the quarter @-@ time record of the season , and the


## 14. Evaluation — Validation Loss and Perplexity
Perplexity = e^loss. Measures how surprised the model is by unseen
text — lower is better.

**Result: Validation Loss 2.11 · Perplexity 8.25**

⚠️ Note: This is lower than expected for a model this size. WikiText-2
is small (~2MB) relative to our 162M parameter model, giving ~80
parameters per training token. The model has partially memorized the
dataset rather than purely generalizing — GPT-2 achieves ~29 perplexity
trained on vastly more data. Our number reflects overfitting to a small
dataset, not superior performance.

In [21]:
import math

def evaluate(model, data_loader, device=DEVICE):
    model.eval()
    total_loss = 0
    total_batches = 0

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            logits = logits.view(-1, 50257)
            y = y.view(-1)

            loss = criterion(logits, y)
            total_loss += loss.item()
            total_batches += 1

    avg_loss = total_loss / total_batches
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity

# build validation dataloader
val_dataset = TextDataset(tokenized["validation"])
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# run after training:
val_loss, val_perplexity = evaluate(model, val_loader)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Perplexity: {val_perplexity:.2f}")

Validation Loss: 2.1106
Validation Perplexity: 8.25
